In [44]:
from tensorflow import keras
from keras import Sequential, Input
from keras.layers import Dense
import numpy as np
import pandas as pd

df = pd.read_csv('train.csv')

x = df.iloc[:, 0:7] 
y = df.iloc[:, 7] 
labels = np.unique(y)
y = y.replace({'Kecimen': 1, 'Besni': 0})

model = Sequential()
model.add(Input(shape=(7,)))
model.add(Dense(12, activation="relu", name="Dense_1"))
model.add(Dense(8, activation="relu", name="Dense_2"))
model.add(Dense(1, activation="sigmoid", name="Dense_3"))

model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
history = model.fit(x, y, epochs=100, batch_size=5, verbose=2)


Epoch 1/100


C:\Users\damin\AppData\Local\Temp\ipykernel_19536\135236717.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = y.replace({'Kecimen': 1, 'Besni': 0})


144/144 - 2s - 14ms/step - accuracy: 0.5319 - loss: 4675.5137
Epoch 2/100
144/144 - 0s - 3ms/step - accuracy: 0.4944 - loss: 0.6952
Epoch 3/100
144/144 - 0s - 3ms/step - accuracy: 0.4944 - loss: 0.6948
Epoch 4/100
144/144 - 1s - 4ms/step - accuracy: 0.4944 - loss: 0.6945
Epoch 5/100
144/144 - 0s - 3ms/step - accuracy: 0.4944 - loss: 0.6942
Epoch 6/100
144/144 - 0s - 3ms/step - accuracy: 0.4944 - loss: 0.6940
Epoch 7/100
144/144 - 0s - 3ms/step - accuracy: 0.4944 - loss: 0.6939
Epoch 8/100
144/144 - 0s - 3ms/step - accuracy: 0.4944 - loss: 0.6937
Epoch 9/100
144/144 - 0s - 3ms/step - accuracy: 0.4944 - loss: 0.6936
Epoch 10/100
144/144 - 0s - 3ms/step - accuracy: 0.4944 - loss: 0.6935
Epoch 11/100
144/144 - 0s - 3ms/step - accuracy: 0.4944 - loss: 0.6934
Epoch 12/100
144/144 - 0s - 3ms/step - accuracy: 0.4944 - loss: 0.6934
Epoch 13/100
144/144 - 0s - 3ms/step - accuracy: 0.4944 - loss: 0.6933
Epoch 14/100
144/144 - 0s - 3ms/step - accuracy: 0.4667 - loss: 0.6932
Epoch 15/100
144/144 - 

In [48]:
from tensorflow import keras
from keras import Sequential, Input
from keras.layers import Dense, Dropout
from keras.callbacks import EarlyStopping
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# ======================
# 1. 학습 데이터 불러오기 및 전처리
# ======================
train_df = pd.read_csv("train.csv")

x = train_df.iloc[:, 0:7].values
y = train_df.iloc[:, 7].replace({'Kecimen': 1, 'Besni': 0}).values

# 데이터 정규화
scaler = StandardScaler()
x = scaler.fit_transform(x)

# 학습용/검증용 분리
x_train, x_val, y_train, y_val = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

# ======================
# 2. 모델 정의
# ======================
model = Sequential([
    Input(shape=(7,)),
    Dense(32, activation="relu", name="Dense_1"),
    Dropout(0.3),
    Dense(16, activation="relu", name="Dense_2"),
    Dropout(0.2),
    Dense(1, activation="sigmoid", name="Output")
])

model.compile(loss="binary_crossentropy",
              optimizer=keras.optimizers.Adam(learning_rate=0.001),
              metrics=["accuracy"])

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# ======================
# 3. 학습
# ======================
history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=100,
    batch_size=8,
    callbacks=[early_stop],
    verbose=2
)

# ======================
# 4. 테스트 데이터 불러오기 및 예측
# ======================
test_df = pd.read_csv("test.csv")
x_test = test_df.iloc[:, 0:7].values
x_test = scaler.transform(x_test)  # 반드시 같은 scaler로 정규화!

# 예측 (sigmoid → 0.5 기준으로 이진화)
y_pred_prob = model.predict(x_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

# 숫자를 다시 문자열로 변환 (제출 형식에 맞게)
y_pred_labels = np.where(y_pred == 1, 'Kecimen', 'Besni')

# ======================
# 5. 제출 파일 생성
# ======================
submission_df = pd.read_csv("submission.csv")
submission_df.dropna(axis=1, inplace=True)

submission_df["Class"] = y_pred_labels
submission_df.to_csv("new_submission.csv", index=False)

print("✅ new_submission.csv 파일 생성 완료!")


Epoch 1/100


C:\Users\damin\AppData\Local\Temp\ipykernel_19536\1781507216.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = train_df.iloc[:, 7].replace({'Kecimen': 1, 'Besni': 0}).values


72/72 - 3s - 37ms/step - accuracy: 0.6250 - loss: 0.6018 - val_accuracy: 0.8958 - val_loss: 0.5467
Epoch 2/100
72/72 - 0s - 6ms/step - accuracy: 0.8403 - loss: 0.4620 - val_accuracy: 0.8819 - val_loss: 0.4554
Epoch 3/100
72/72 - 0s - 6ms/step - accuracy: 0.8559 - loss: 0.3702 - val_accuracy: 0.8750 - val_loss: 0.4336
Epoch 4/100
72/72 - 0s - 6ms/step - accuracy: 0.8715 - loss: 0.3562 - val_accuracy: 0.8819 - val_loss: 0.4335
Epoch 5/100
72/72 - 0s - 6ms/step - accuracy: 0.8646 - loss: 0.3372 - val_accuracy: 0.8750 - val_loss: 0.4419
Epoch 6/100
72/72 - 0s - 6ms/step - accuracy: 0.8646 - loss: 0.3584 - val_accuracy: 0.8819 - val_loss: 0.4298
Epoch 7/100
72/72 - 0s - 6ms/step - accuracy: 0.8646 - loss: 0.3461 - val_accuracy: 0.8889 - val_loss: 0.4287
Epoch 8/100
72/72 - 0s - 6ms/step - accuracy: 0.8611 - loss: 0.3525 - val_accuracy: 0.8819 - val_loss: 0.4297
Epoch 9/100
72/72 - 0s - 6ms/step - accuracy: 0.8715 - loss: 0.3461 - val_accuracy: 0.8889 - val_loss: 0.4262
Epoch 10/100
72/72 - 

In [56]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LeakyReLU, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

# ======================
# 1. 학습 데이터 불러오기 및 전처리
# ======================
train_df = pd.read_csv("train.csv")
x = train_df.iloc[:, 0:7].values
y = train_df.iloc[:, 7].replace({'Kecimen': 1, 'Besni': 0}).values

# MinMaxScaler 적용
scaler = MinMaxScaler()
x = scaler.fit_transform(x)

# Train/Validation 분리
x_train, x_val, y_train, y_val = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=42
)

# ======================
# 2. 모델 정의
# ======================
model = Sequential([
    Input(shape=(7,)),
    Dense(64),
    BatchNormalization(),
    LeakyReLU(0.1),
    Dropout(0.4),

    Dense(32),
    BatchNormalization(),
    LeakyReLU(0.1),
    Dropout(0.3),

    Dense(16),
    BatchNormalization(),
    LeakyReLU(0.1),

    Dense(1, activation='sigmoid')
])

# ======================
# 3. 컴파일 + 콜백
# ======================
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)

# ======================
# 4. 학습
# ======================
history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=300,
    batch_size=16,
    callbacks=[early_stop, reduce_lr],
    verbose=2
)

# ======================
# 5. 테스트 데이터 예측
# ======================
test_df = pd.read_csv("test.csv")
x_test = test_df.iloc[:, 0:7].values
x_test = scaler.transform(x_test)

y_pred_prob = model.predict(x_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()
y_pred_labels = np.where(y_pred==1, 'Kecimen', 'Besni')

# ======================
# 6. 제출 파일 생성
# ======================
submission_df = pd.read_csv("submission.csv")
submission_df.dropna(axis=1, inplace=True)
submission_df["Class"] = y_pred_labels
submission_df.to_csv("new_submission.csv", index=False)

# ======================
# 7. 검증 정확도 확인
# ======================
val_loss, val_acc = model.evaluate(x_val, y_val, verbose=0)
print(f"📈 검증 정확도: {val_acc:.4f}")


Epoch 1/300


C:\Users\damin\AppData\Local\Temp\ipykernel_19536\2742278117.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = train_df.iloc[:, 7].replace({'Kecimen': 1, 'Besni': 0}).values


36/36 - 5s - 136ms/step - accuracy: 0.7066 - loss: 0.5520 - val_accuracy: 0.5556 - val_loss: 0.6526 - learning_rate: 0.0010
Epoch 2/300
36/36 - 0s - 8ms/step - accuracy: 0.7969 - loss: 0.4525 - val_accuracy: 0.5556 - val_loss: 0.6406 - learning_rate: 0.0010
Epoch 3/300
36/36 - 0s - 8ms/step - accuracy: 0.8108 - loss: 0.4123 - val_accuracy: 0.5972 - val_loss: 0.6197 - learning_rate: 0.0010
Epoch 4/300
36/36 - 0s - 8ms/step - accuracy: 0.8212 - loss: 0.4123 - val_accuracy: 0.6875 - val_loss: 0.6000 - learning_rate: 0.0010
Epoch 5/300
36/36 - 0s - 8ms/step - accuracy: 0.8229 - loss: 0.3991 - val_accuracy: 0.7500 - val_loss: 0.5812 - learning_rate: 0.0010
Epoch 6/300
36/36 - 0s - 8ms/step - accuracy: 0.8212 - loss: 0.4080 - val_accuracy: 0.8333 - val_loss: 0.5590 - learning_rate: 0.0010
Epoch 7/300
36/36 - 0s - 8ms/step - accuracy: 0.8368 - loss: 0.3779 - val_accuracy: 0.8472 - val_loss: 0.5329 - learning_rate: 0.0010
Epoch 8/300
36/36 - 0s - 8ms/step - accuracy: 0.8264 - loss: 0.3956 - va